# Set Up

In [27]:
!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu


In [28]:
!pip install ipywidgets

In [29]:
!pip install -qqqqqqqq arize-otel arize agno openai openinference-instrumentation-agno openinference-instrumentation-openai httpx chromadb sentence-transformers arize-phoenix

In [30]:
import os
from getpass import getpass

os.environ["ARIZE_SPACE_ID"] = globals().get("ARIZE_SPACE_ID") or getpass("🔑 Enter your Arize Space ID: ")

os.environ["ARIZE_API_KEY"] = globals().get("ARIZE_API_KEY") or getpass("🔑 Enter your Arize API Key: ")

os.environ["OPENAI_API_KEY"] = globals().get("OPENAI_API_KEY") or getpass("🔑 Enter your OpenAI API Key: ")

os.environ["TAVILY_API_KEY"] = globals().get("TAVILY_API_KEY") or getpass("🔑 Enter your Tavily API Key: ")

In [31]:
from arize.otel import register
from openinference.instrumentation.openai import OpenAIInstrumentor
from openinference.instrumentation.agno import AgnoInstrumentor

model_id = "evaluate-travel-agent"
tracer_provider = register(
    space_id=os.getenv("ARIZE_SPACE_ID"),
    api_key=os.getenv("ARIZE_API_KEY"),
    project_name=model_id,
    set_global_tracer_provider=True
)
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)
AgnoInstrumentor().instrument(tracer_provider=tracer_provider)

Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


🔭 OpenTelemetry Tracing Details 🔭
|  Arize Project: evaluate-travel-agent
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



# Define Tools

In [41]:
# --- Helper functions for tools ---
import httpx
from opentelemetry import trace

tracer = trace.get_tracer(__name__)

@tracer.chain(name="search-api")
def _search_api(query: str) -> str | None:
    """Try Tavily search first, fall back to None."""
    tavily_key = os.getenv("TAVILY_API_KEY")
    if not tavily_key:
        return None
    try:
        resp = httpx.post(
            "https://api.tavily.com/search",
            json={
                "api_key": tavily_key,
                "query": query,
                "max_results": 3,
                "search_depth": "basic",
                "include_answer": True,
            },
            timeout=8,
        )
        data = resp.json()
        answer = data.get("answer") or ""
        snippets = [r.get("content", "") for r in data.get("results", [])]
        combined = " ".join([answer] + snippets).strip()
        return combined[:400] if combined else None
    except Exception:
        return None

def _compact(text: str, limit: int = 200) -> str:
    """Compact text for cleaner outputs."""
    cleaned = " ".join(text.split())
    return cleaned if len(cleaned) <= limit else cleaned[:limit].rsplit(" ", 1)[0]


In [42]:
# --- APIs for Essential Info Tool ---
import httpx
from urllib.parse import quote
from typing import Optional

@tracer.chain(name="wiki-summary-api")
def _wiki_summary(dest: str) -> str:
    if not dest:
        return ""
    encoded_dest = quote(dest)

    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{encoded_dest}"
    HEADERS = { 'User-Agent': 'MyArizeApp/1.0 (ExampleContac@example.com)'}

    try:
        r = httpx.get(url, headers = HEADERS, timeout=5)
        r.raise_for_status()

        data = r.json().get("extract")
        return data if data else ""

    except httpx.HTTPStatusError as e:
        if e.response.status_code == 404:
            return ""
        return ""
    except httpx.RequestError as e:
        return ""
    except Exception as e:
        return ""

@tracer.chain(name="weather-api")
def _weather(dest):
    g = httpx.get(f"https://geocoding-api.open-meteo.com/v1/search?name={dest}")
    if g.status_code != 200 or not g.json().get("results"):
        return ""
    lat, lon = g.json()["results"][0]["latitude"], g.json()["results"][0]["longitude"]
    w = httpx.get(f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true").json()
    cw = w.get("current_weather", {})
    return f"Weather now: {cw.get('temperature')}°C, wind {cw.get('windspeed')} km/h."

# Create RAG System for Manufacturing Knowledge

In [34]:
import sys, site, importlib.util

print("Python:", sys.version)
print("Executable:", sys.executable)

print("\nUser site:", site.getusersitepackages())
print("System site:", site.getsitepackages() if hasattr(site, "getsitepackages") else "n/a")

print("\nTorch spec:", importlib.util.find_spec("torch"))
print("Transformers spec:", importlib.util.find_spec("transformers"))


Python: 3.10.12 (main, Jan  8 2026, 06:52:19) [GCC 11.4.0]
Executable: /home/echorosal/Documents/repo/agents/tutorials/python/llm/agents/agent-mastery-course/ragenv/bin/python

User site: /home/echorosal/.local/lib/python3.10/site-packages
System site: ['/home/echorosal/Documents/repo/agents/tutorials/python/llm/agents/agent-mastery-course/ragenv/lib/python3.10/site-packages', '/home/echorosal/Documents/repo/agents/tutorials/python/llm/agents/agent-mastery-course/ragenv/local/lib/python3.10/dist-packages', '/home/echorosal/Documents/repo/agents/tutorials/python/llm/agents/agent-mastery-course/ragenv/lib/python3/dist-packages', '/home/echorosal/Documents/repo/agents/tutorials/python/llm/agents/agent-mastery-course/ragenv/lib/python3.10/dist-packages']

Torch spec: ModuleSpec(name='torch', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7f565085c1f0>, origin='/home/echorosal/Documents/repo/agents/tutorials/python/llm/agents/agent-mastery-course/ragenv/lib/python3.10/site-

In [35]:
import chromadb
from sentence_transformers import SentenceTransformer

chroma_client = chromadb.Client()
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Create the LLMaf Contextual Cards collection
collection = chroma_client.create_collection(
    name="llmaf_context_cards",
    metadata={"hnsw:space": "cosine"}
)

print("✅ LLMaf RAG system initialized for Industrial Contextual Cards")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


InternalError: Collection [llmaf_context_cards] already exists

In [ ]:
import json
import os

def load_and_index_llmaf_context():
    # Define the files you uploaded
    card_files = [
        'data_sheet_card.json', 
        'domain_card.json', 
        'model_card.json', 
        'profile_card.json',
        'ruleset.json'
    ]

    documents = []
    metadatas = []
    ids = []

    for file_name in card_files:
        if not os.path.exists(file_name):
            print(f"⚠️ {file_name} not found, skipping...")
            continue
            
        with open(file_name, 'r') as f:
            data = json.load(f)
            
        # 1. Create a Primary Entry for the entire Card
        # This helps the agent understand the "big picture" (e.g., Process Domain)
        card_name = data.get('name', file_name.replace('.json', '').title())
        
        # We create a rich text summary for the embedding model to "understand" the card
        summary_text = f"Contextual Card: {card_name}. "
        if 'description' in data:
            summary_text += f"Description: {data['description']}"
        
        # We store the full JSON string in metadata so the agent can read the whole thing
        documents.append(f"{summary_text}\nContent: {json.dumps(data)}")
        metadatas.append({
            "source": file_name,
            "card_type": card_name,
            "granularity": "full_card"
        })
        ids.append(f"card_{file_name.split('.')[0]}")

        # 2. Special Handling for Rules (Granular Indexing)
        # In LLMaf, we need to find specific rules. We index each rule as its own document.
        if file_name == 'ruleset.json' and 'rules' in data:
            for rule in data['rules']:
                rule_body = rule.get('Body of the rule', '')
                rule_id = str(rule.get('rule_id'))
                
                # The text used for vector search
                rule_doc = f"Manufacturing Rule {rule_id}: {rule_body} -> Predicts Class {rule.get('Head of the rule')}"
                
                documents.append(rule_doc)
                metadatas.append({
                    "source": file_name,
                    "card_type": "Individual Rule",
                    "rule_id": rule_id,
                    "granularity": "granular"
                })
                ids.append(f"rule_{rule_id}")

    # Add all at once to ChromaDB
    collection.add(
        documents=documents,
        metadatas=metadatas,
        ids=ids
    )

    print(f"✅ LLMaf Database Updated: Indexed {len(documents)} context elements.")
    return len(documents)

# Run the loader
num_entries = load_and_index_llmaf_context()

✅ LLMaf Database Updated: Indexed 11 context elements.


In [ ]:
results = collection.get(limit=5, include=['documents', 'metadatas'])
for i in range(len(results['ids'])):
    print(f"ID: {results['ids'][i]}")
    print(f"Type: {results['metadatas'][i]['card_type']}")
    print(f"Preview: {results['documents'][i][:100]}...\n")

ID: card_data_sheet_card
Type: Dataset Card
Preview: Contextual Card: Dataset Card. 
Content: {"name": "Dataset Card", "author": {"name": "Josue Obregon"...

ID: card_domain_card
Type: Domain Card
Preview: Contextual Card: Domain Card. Description: Injection moulding is the most widely used process in the...

ID: card_model_card
Type: Model Card
Preview: Contextual Card: Model Card. 
Content: {"name": "Model Card", "author": {"name": "Josue Obregon", "a...

ID: card_profile_card
Type: Profile Card
Preview: Contextual Card: Profile Card. 
Content: {"name": "Profile Card", "author": {"name": "Josue Obregon"...

ID: card_ruleset
Type: Ruleset
Preview: Contextual Card: Ruleset. 
Content: {"number of rules": 6, "accuracy": 1.0, "f-measure": 1.0, "sampl...



In [ ]:
from sentence_transformers import SentenceTransformer
from openinference.semconv.trace import SpanAttributes

# Initialize embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

def get_manufacturing_context(query: str, user_profile: str = "Operator") -> str:
    """
    Retrieves LLMaf Contextual Cards (Domain, Model, Profile, Data Sheet) 
    and specific Rules to explain manufacturing quality predictions.
    """
    print(f"🔍 RAG Tool Triggered: Searching for '{query}' for profile '{user_profile}'...")

    with tracer.start_as_current_span(name="LLMaf_RAG", attributes={SpanAttributes.OPENINFERENCE_SPAN_KIND: "RETRIEVER"}) as span:
        
        # 1. Construct semantic query (e.g., "Operator explanation for Dosage_time_max")
        search_query = f"{user_profile} perspective on {query}"
        span.set_attribute(SpanAttributes.INPUT_VALUE, search_query)

        # 2. Embed the query
        query_embedding = embedding_model.encode([search_query])

        # 3. Search the 'llmaf_context_cards' collection
        results = collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=4  # Retrieve enough to get the Rule + Domain + Profile cards
        )

        if not results or not results.get("documents") or not results["documents"][0]:
            return "No specific manufacturing context found for this query."

        # 4. Extract and Format Results
        retrieved_docs = results["documents"][0]
        retrieved_meta = results["metadatas"][0]
        
        context_blocks = []
        for doc, meta in zip(retrieved_docs, retrieved_meta):
            source = meta.get("source", "Unknown Card")
            card_type = meta.get("card_type", "Context")
            
            # Format as a structured block for the LLM to read
            block = f"--- SOURCE: {source} ({card_type}) ---\n{doc}\n"
            context_blocks.append(block)

        response = "Relevant Manufacturing Context:\n\n" + "\n".join(context_blocks)
        span.set_attribute(SpanAttributes.OUTPUT_VALUE, response)
        
        print(f"✅ Found {len(results['documents'][0])} relevant context snippets.")
        return response

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Define Agent

In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

llmaf_agent = Agent(
    name="LLMaf-Analyst",
    model=OpenAIChat(id="gpt-4o"),
    instructions=[
        "You are an intelligent assistant for quality management in manufacturing.",
        "STRICT SCOPE: If a question is unrelated to manufacturing quality, reply: 'I’m specialized in manufacturing quality prediction and cannot assist with that topic.'",
        
        "1. ROLE ADAPTATION: Use 'get_manufacturing_context' to check the Profile Card.",
        "   - Operator: Action-focused, brief, level 3 domain physics.",
        "   - Manager: Include KPI/impact (yield, scrap) without inventing numbers.",
        "   - Data Scientist: Include technical metrics (Coverage, Support) from the ruleset.",

        "2. RULE INVERSION (PREVENTION): If intent is 'avoid' and rule head=1 (defective):",
        "   - Var >= T -> keep below T; Var > T -> keep at or below T.",
        "   - Var <= T -> keep above T; Var < T -> keep at or above T.",
        
        "3. OUTPUT SHAPE (MANDATORY):",
        "   - Step 1: Action bullets using display labels and thresholds.",
        "   - Step 2: Short explanation grounded in Domain Card phases (Filling, Packing, etc.).",
        "   - Step 3: State direction using only: 'increases', 'decreases', or 'no effect'.",
        
        "4. SEMANTIC FIDELITY:",
        "   - Use PARAMETER_DISPLAY_MAP for all names.",
        "   - Head=1 -> 'defective product'; Head=0 -> 'normal product'.",
        "   - No Rule IDs, no JSON keys, no prefaces."
    ],
    tools=[get_manufacturing_context],
    markdown=True
)

In [36]:
#03032026

from agno.agent import Agent
from agno.models.openai import OpenAIChat

intent_classifier = Agent(
    name="Intent-Classifier",
    role="Classify manufacturing queries into specific intent categories.",
    model=OpenAIChat(id="gpt-4o"), # You can swap to your local model later
    instructions=[
        "Classify the user's input into exactly one of these categories in order of priority:",
        "1. OutOfScope: Unrelated to manufacturing quality.",
        "2. OperationalActions: Asking for adjustments, prevention, or line actions.",
        "3. BusinessInsights: Focus on KPIs, yield, scrap, coverage, or support metrics.",
        "4. ModelInsights: Questions about rule mechanics, thresholds, or the pipeline.",
        "5. Explanations: Asking 'why' or the meaning of a specific rule/condition.",
        "Output ONLY the category name."
    ],
)


In [37]:
entity_extractor = Agent(
    name="Entity-Extractor",
    role="Extract manufacturing variables and numerical thresholds.",
    model=OpenAIChat(id="gpt-4o"),
    instructions=[
        "Extract all manufacturing parameters (e.g., 'Injection Pressure') and their values.",
        "Format the output as a clean list of Variable: Value.",
        "If no values are found, return 'None'."
    ],
)

In [53]:
llmaf_analyst = Agent(
    name="LLMaf-Analyst",
    role="Quality Management Specialist",
    model=OpenAIChat(id="gpt-4o"),
    instructions=[
        "You are the core reasoning engine of the LLMaf framework.",
        "ADAPT YOUR OUTPUT FORMAT BASED ON THE DETECTED INTENT:",

        "1. IF Intent == 'OperationalActions':",
        "   - Use ACTION BULLETS for immediate floor-level adjustments.",
        "   - Apply RULE INVERSION: If intent is 'avoid' and rule head=1 (defective), flip logic (e.g., > becomes <=).",
        "   - End with a 'Direction' statement (increases/decreases).",

        "2. IF Intent == 'BusinessInsights':",
        "   - Focus on KPI impacts (Yield, Scrap Rate, Cost of Quality).",
        "   - Use a STRATEGIC TONE. Relate rules to production volume and stability.",
        "   - Format: Brief summary followed by a 'Impact Analysis' section.",

        "3. IF Intent == 'ModelInsights':",
        "   - Output TECHNICAL DETAILS: Include Coverage, Support, and Rule Ordering.",
        "   - Explain the threshold logic and why this specific rule was triggered in the pipeline.",
        "   - Format: Technical specification table or list.",

        "4. IF Intent == 'Explanations':",
        "   - Use NARRATIVE PARAGRAPHS grounded in Domain Physics.",
        "   - Explain the 'Why' behind the sensor reading (e.g., 'High pressure leads to material flash because...').",
        "   - Format: Plain prose, easy to read for non-experts.",

        "CRITICAL: Always use PARAMETER_DISPLAY_MAP for names. No raw JSON or Rule IDs."
    ],
    tools=[get_manufacturing_context],
    markdown=True
)

In [43]:
pip show agno

Name: agno
Version: 2.4.8
Summary: Agno: a lightweight library for building Multi-Agent Systems
Home-page: https://agno.com
Author: 
Author-email: Ashpreet Bedi <ashpreet@agno.com>
License: 
Location: /home/echorosal/Documents/repo/agents/tutorials/python/llm/agents/agent-mastery-course/ragenv/lib/python3.10/site-packages
Requires: docstring-parser, gitpython, h11, httpx, packaging, pydantic, pydantic-settings, python-dotenv, python-multipart, pyyaml, rich, typer, typing-extensions
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [56]:
from agno.workflow import Workflow

class LLMafFramework(Workflow):
    def run(self, message: str):
        # STEP A: Intent Classification
        classification = intent_classifier.run(message)
        intent = classification.content.strip()
        
        # STEP B: Scope Guardrail
        if "OutOfScope" in intent:
            # We return a dictionary so the CSV stays consistent
            return {
                "intent": "OutOfScope",
                "content": "I’m specialized in manufacturing quality prediction and cannot assist with that topic."
            }

        # STEP C: Entity Extraction
        extraction = entity_extractor.run(message)
        entities = extraction.content

        # STEP D: Final Routing with Format Instructions
        # We explicitly tell the analyst which intent was found
        contextual_prompt = (
            f"ACTUAL INTENT: {intent}\n"
            f"EXTRACTED ENTITIES: {entities}\n"
            f"USER QUESTION: {message}\n\n"
            f"Please provide the response following the specific format for {intent}."
        )
        
        final_result = llmaf_agent.run(contextual_prompt)
        
        # RETURN AS DICTIONARY FOR THE CSV BATCH RUN
        return {
            "intent": intent,
            "content": final_result.content
        }

In [ ]:
from agno.workflow import Workflow

class LLMafFramework(Workflow):
    def run(self, message: str):
        # STEP A: Intent Classification
        classification = intent_classifier.run(message)
        intent = classification.content.strip()
        
        # STEP B: Scope Guardrail
        if "OutOfScope" in intent:
            # We return a dictionary so the CSV stays consistent
            return {
                "intent": "OutOfScope",
                "content": "I’m specialized in manufacturing quality prediction and cannot assist with that topic."
            }

        # STEP C: Entity Extraction
        extraction = entity_extractor.run(message)
        entities = extraction.content

        # STEP D: Final Routing with Format Instructions
        # We explicitly tell the analyst which intent was found
        contextual_prompt = (
            f"ACTUAL INTENT: {intent}\n"
            f"EXTRACTED ENTITIES: {entities}\n"
            f"USER QUESTION: {message}\n\n"
            f"Please provide the response following the specific format for {intent}."
        )
        
        final_result = llmaf_agent.run(contextual_prompt)
        
        # RETURN AS DICTIONARY FOR THE CSV BATCH RUN
        return {
            "intent": intent,
            "content": final_result.content
        }

In [49]:
from IPython.display import display, HTML

def run_llmaf_white_style(role, question):
    # 1. EXECUTE THE WORKFLOW
    # Since your workflow's run() method returns the final content string
    # We call the full system, not just the single agent.
    full_query = f"ROLE: {role}. QUESTION: {question}"
    final_output = llmaf_system.run(full_query)
    
    # 2. PRE-PROCESS for HTML display
    content = final_output
    content = content.replace('\n', '<br>')
    # Simple markdown-to-HTML replacements
    content = content.replace('**', '<b>').replace('###', '<h4>')

    # 3. HTML and CSS
    html_output = f"""
    <div style="background-color: #ffffff; color: #2c3e50; padding: 30px; border: 1px solid #dcdde1; 
                border-radius: 8px; font-family: sans-serif; max-width: 850px; margin: 20px auto; 
                box-shadow: 0 4px 15px rgba(0,0,0,0.05);">
        
        <div style="border: 2px solid #2ecc71; border-radius: 6px; padding: 15px; margin-bottom: 25px; position: relative;">
            <span style="position: absolute; top: -12px; left: 15px; background: white; padding: 0 10px; color: #2ecc71; font-weight: bold; font-size: 0.85em;">
                LLMaf Input Context
            </span>
            <strong style="color: #16a085;">Role:</strong> {role}<br>
            <strong style="color: #16a085;">Question:</strong> {question}
        </div>

        <div style="margin-bottom: 20px; font-size: 0.8em; color: #7f8c8d; border-bottom: 1px solid #eee; padding-bottom: 10px;">
            <span style="background: #ecf0f1; padding: 3px 8px; border-radius: 4px; margin-right: 10px;">✔ Intent Classified</span>
            <span style="background: #ecf0f1; padding: 3px 8px; border-radius: 4px; margin-right: 10px;">✔ Entities Extracted</span>
            <span style="background: #ecf0f1; padding: 3px 8px; border-radius: 4px;">✔ Rule Matched</span>
        </div>

        <div style="border: 2px solid #3498db; border-radius: 6px; padding: 20px; position: relative;">
            <span style="position: absolute; top: -12px; left: 15px; background: white; padding: 0 10px; color: #3498db; font-weight: bold; font-size: 0.85em;">
                LLMaf Explanation (Prescriptive)
            </span>
            <div style="line-height: 1.7;">
                {content}
            </div>
        </div>
        
        <div style="margin-top: 15px; text-align: right; font-size: 0.75em; color: #bdc3c7;">
            Agentic Pipeline: Intent -> Entities -> LLMaf Retrieval
        </div>
    </div>
    """
    
    display(HTML(html_output))

# --- Run the Test ---
run_llmaf_white_style("Operator", "How can I avoid defects related to Rule 1?")

--- [PHASE 1: INTENT] --- 
Detected: OperationalActions

--- [PHASE 2: ENTITIES] --- 
Found: None

🔍 RAG Tool Triggered: Searching for 'Rule 1' for profile 'Operator'...
✅ Found 4 relevant context snippets.


In [62]:
run_llmaf_white_style("Operator", "What should I check for KPI for weekly monitoring progress?")

--- [PHASE 1: INTENT] --- 
Detected: BusinessInsights

--- [PHASE 2: ENTITIES] --- 
Found: None

🔍 RAG Tool Triggered: Searching for 'KPI weekly monitoring' for profile 'Operator'...
✅ Found 4 relevant context snippets.


In [64]:
run_llmaf_white_style("Operator", "If the Maximum Injection Pressure drops below 1728.7 as seen in Rule 5, what physical adjustment should I check on the machine first?")

--- [PHASE 1: INTENT] --- 
Detected: OperationalActions

--- [PHASE 2: ENTITIES] --- 
Found: - Maximum Injection Pressure: 1728.7

🔍 RAG Tool Triggered: Searching for 'Maximum Injection Pressure' for profile 'Operator'...
✅ Found 4 relevant context snippets.


In [60]:
run_llmaf_white_style("Operator", "What portion of defects comes from Rule 5?")

--- [PHASE 1: INTENT] --- 
Detected: Explanations

--- [PHASE 2: ENTITIES] --- 
Found: None

🔍 RAG Tool Triggered: Searching for 'Rule 5' for profile 'Operator'...
✅ Found 4 relevant context snippets.


In [65]:
run_llmaf_white_style("Manager", "What supplier factors affect Rule 4?")

--- [PHASE 1: INTENT] --- 
Detected: Explanations

--- [PHASE 2: ENTITIES] --- 
Found: None

🔍 RAG Tool Triggered: Searching for 'Rule 4' for profile 'Manager'...
✅ Found 4 relevant context snippets.


In [61]:
run_llmaf_white_style("Operator", "Can you explain what does 'Material Cushion Median' mean as defined in Rule 3?")

--- [PHASE 1: INTENT] --- 
Detected: Explanations

--- [PHASE 2: ENTITIES] --- 
Found: None

🔍 RAG Tool Triggered: Searching for 'Material Cushion Median' for profile 'Operator'...
✅ Found 4 relevant context snippets.


In [66]:
run_llmaf_white_style("manager", "Where should we allocate maintenance resources to reduce overall defects?")



--- [PHASE 1: INTENT] --- 
Detected: OperationalActions

--- [PHASE 2: ENTITIES] --- 
Found: None

🔍 RAG Tool Triggered: Searching for 'maintenance resource allocation to reduce defects' for profile 'manager'...
✅ Found 4 relevant context snippets.


In [67]:
run_llmaf_white_style("Data Scientist", "Is the ordered list structure sensitive to the sequence of Rules 1 through 5, or are they mutually exclusive?")

--- [PHASE 1: INTENT] --- 
Detected: ModelInsights

--- [PHASE 2: ENTITIES] --- 
Found: None

🔍 RAG Tool Triggered: Searching for 'Rules 1 to 5 sequence sensitivity' for profile 'Data Scientist'...
✅ Found 4 relevant context snippets.


In [ ]:

# 2. How to run a single query with a separated Role
def process_manufacturing_query(user_role: str, user_question: str):
    # We combine the role into the query so the agent starts with the right persona
    full_prompt = f"Role: {user_role}. Question: {user_question}"
    
    print(f"--- Processing for {user_role} ---")
    llmaf_agent.print_response(full_prompt)

# --- Test Case 1: Manager ---
process_manufacturing_query(
    user_role="Manager", 
    user_question="What are the conditions that lead to defects in Rule 5?"
)

# --- Test Case 2: Data Scientist ---
process_manufacturing_query(
    user_role="Data Scientist", 
    user_question="Explain the technical performance of Rule 1."
)

In [57]:
import pandas as pd
import csv
from tqdm import tqdm

def run_batch_evaluation(input_csv, output_csv, limit=55):
    df = pd.read_csv(input_csv)
    if limit:
        df = df.head(limit)
    
    results = []
    print(f"🚀 Processing {len(df)} questions for LLMaf Thesis Evaluation...")
    
    for index, row in tqdm(df.iterrows(), total=len(df)):
        role = row['Role']
        question = row['User Question']
        full_query = f"ROLE: {role}. QUESTION: {question}"
        
        try:
            # Call the workflow - it now returns a dictionary
            response_data = llmaf_system.run(full_query)
            
            results.append({
                "Role": role,
                "User_Question": question,
                "Detected_Intent": response_data["intent"], # Column 3
                "LLMaf_Final_Response": response_data["content"] # Column 4
            })
        except Exception as e:
            results.append({
                "Role": role, "User_Question": question,
                "Detected_Intent": "ERROR", "LLMaf_Final_Response": str(e)
            })

    # Save with QUOTE_ALL to prevent "Comma Splitting" issues
    results_df = pd.DataFrame(results)
    results_df.to_csv(
        output_csv, 
        index=False, 
        quoting=csv.QUOTE_ALL, 
        encoding='utf-8-sig' # Ensures Korean text displays correctly in Excel
    )
    print(f"✅ Success! Data saved to {output_csv}")
    return results_df

# Run it!
results = run_batch_evaluation('intelligent_agent_user_questions_stage1.csv', 'llmaf_results_formatted.csv')

🚀 Processing 54 questions for LLMaf Thesis Evaluation...


  0%|          | 0/54 [00:00<?, ?it/s]

--- [PHASE 1: INTENT] --- 
Detected: Explanations

--- [PHASE 2: ENTITIES] --- 
Found: None

🔍 RAG Tool Triggered: Searching for 'Material Cushion Median' for profile 'Operator'...
✅ Found 4 relevant context snippets.


  2%|▏         | 1/54 [00:07<07:02,  7.97s/it]

--- [PHASE 1: INTENT] --- 
Detected: Explanations

--- [PHASE 2: ENTITIES] --- 
Found: - Dosage time max: High

🔍 RAG Tool Triggered: Searching for 'Dosage_time_max impact on defect risk' for profile 'Operator'...
✅ Found 4 relevant context snippets.


...

--- [PHASE 1: INTENT] --- 
Detected: OutOfScope

--- [PHASE 1: INTENT] --- 
Detected: ModelInsights

--- [PHASE 2: ENTITIES] --- 
Found: None

🔍 RAG Tool Triggered: Searching for 'Empty default rule bias' for profile 'data scientist'...
✅ Found 4 relevant context snippets.


 98%|█████████▊| 53/54 [06:20<00:04,  4.59s/it]

--- [PHASE 1: INTENT] --- 
Detected: OutOfScope



100%|██████████| 54/54 [06:22<00:00,  7.08s/it]

--- [PHASE 1: INTENT] --- 
Detected: OutOfScope

✅ Success! Data saved to llmaf_results_formatted.csv


In [59]:
import pandas as pd
import csv
from tqdm import tqdm

# 1. Load your 55 questions
df = pd.read_csv('intelligent_agent_user_questions_stage1.csv')
results = []

print(f"🚀 Processing {len(df)} questions...")

for index, row in tqdm(df.iterrows(), total=len(df)):
    role = row['Role']
    question = row['User Question']
    
    try:
        # 2. Call your modified workflow
        # Because Step 1 is done, 'output' is now a DICTIONARY
        output = llmaf_system.run(f"ROLE: {role}. QUESTION: {question}")
        
        # 3. UNPACK the dictionary into separate keys for the CSV
        results.append({
            "Role": role,
            "User_Question": question,
            "Detected_Intent": output["intent"],   # Clean column 3
            "LLMaf_Response": output["content"]    # Clean column 4
        })
    except Exception as e:
        results.append({
            "Role": role, "User_Question": question,
            "Detected_Intent": "ERROR", "LLMaf_Response": str(e)
        })

# 4. SAVE WITH PROTECTIVE QUOTING
# This prevents commas inside the AI response from breaking the CSV
final_df = pd.DataFrame(results)
final_df.to_csv(
    'llmaf_stage1_results_final.csv', 
    index=False, 
    quoting=csv.QUOTE_ALL, 
    encoding='utf-8-sig'
)

print("✅ DONE! Download 'llmaf_stage1_results_final.csv' and open it in Excel.")

🚀 Processing 54 questions...


  0%|          | 0/54 [00:00<?, ?it/s]

--- [PHASE 1: INTENT] --- 
Detected: Explanations

--- [PHASE 2: ENTITIES] --- 
Found: None

🔍 RAG Tool Triggered: Searching for 'Material Cushion Median' for profile 'Operator'...
✅ Found 4 relevant context snippets.


  2%|▏         | 1/54 [00:07<06:57,  7.88s/it]

--- [PHASE 1: INTENT] --- 
Detected: Explanations

--- [PHASE 2: ENTITIES] --- 
Found: - Dosage Time Max: High

🔍 RAG Tool Triggered: Searching for 'Why does high Dosage_time_max increase defect risk?' for profile 'Operator'...
✅ Found 4 relevant context snippets.


...

--- [PHASE 1: INTENT] --- 
Detected: Explanations

--- [PHASE 2: ENTITIES] --- 
Found: None

🔍 RAG Tool Triggered: Searching for 'empty default rule' for profile 'Data Scientist'...
✅ Found 4 relevant context snippets.


 98%|█████████▊| 53/54 [05:29<00:05,  5.69s/it]

--- [PHASE 1: INTENT] --- 
Detected: OutOfScope



100%|██████████| 54/54 [05:31<00:00,  6.13s/it]

--- [PHASE 1: INTENT] --- 
Detected: OutOfScope

✅ DONE! Download 'llmaf_stage1_results_final.csv' and open it in Excel.


In [52]:
# Display the first 3 results in your fancy UI
for i in range(3):
    role = results.iloc[i]['Role']
    question = results.iloc[i]['Question']
    # We call the display function we built earlier
    run_llmaf_white_style(role, question)

--- [PHASE 1: INTENT] --- 
Detected: Explanations

--- [PHASE 2: ENTITIES] --- 
Found: None

🔍 RAG Tool Triggered: Searching for 'Material Cushion Median' for profile 'Operator'...
✅ Found 4 relevant context snippets.


--- [PHASE 1: INTENT] --- 
Detected: Explanations

--- [PHASE 2: ENTITIES] --- 
Found: None

🔍 RAG Tool Triggered: Searching for 'Dosage_time_max defect risk' for profile 'Operator'...
✅ Found 4 relevant context snippets.


--- [PHASE 1: INTENT] --- 
Detected: Explanations

--- [PHASE 2: ENTITIES] --- 
Found: None

🔍 RAG Tool Triggered: Searching for 'rules explanation' for profile 'Operator'...
✅ Found 4 relevant context snippets.
